# Play Making:

- Looking at the pass play and routes for a need to have play such as 2-point conversion or fourth down play within the 2-minute drill data. 
- We could look at the characteristics that give a higher percentage of completion. 
- Gives coaches an effective way of making a decision on what is the best chance of completion for each given defense and situation.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
supp = pd.read_csv('C:\\Users\\jrzem\\Downloads\\NFL-Big-Data-Bowl-2026-Analytics-Challenge\\data\\raw\\supplementary_data.csv')

# Parse game clock
def parse_clock(clock_str):
    try:
        parts = str(clock_str).split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except:
        return None

supp['seconds_remaining'] = supp['game_clock'].apply(parse_clock)
supp['is_complete'] = (supp['pass_result'] == 'C').astype(int)

# =============================================================================
# FILTER: 4TH DOWN AND 2-POINT CONVERSIONS
# =============================================================================
print("="*60)
print("4TH DOWN & 2-POINT CONVERSION ANALYSIS")
print("="*60)

# 4th down plays
fourth_down = supp[supp['down'] == 4].copy()

# 2-point conversions (typically show as specific yardline/situation)
# Usually from 2-yard line after TD, often marked in play description
two_pt = supp[supp['play_description'].str.contains('TWO-POINT|2-point|two point', case=False, na=False)].copy()

print(f"\n4th Down Plays: {len(fourth_down)}")
print(f"2-Point Conversions: {len(two_pt)}")
print(f"Combined unique plays: {len(pd.concat([fourth_down, two_pt]).drop_duplicates())}")

# Combine for analysis
critical_plays = pd.concat([fourth_down, two_pt]).drop_duplicates()
critical_plays['play_type'] = critical_plays['down'].apply(lambda x: '4th Down' if x == 4 else '2-PT Conversion')
print(f"\nOverall completion rate: {critical_plays['is_complete'].mean()*100:.1f}%")

4TH DOWN & 2-POINT CONVERSION ANALYSIS

4th Down Plays: 491
2-Point Conversions: 0
Combined unique plays: 491

Overall completion rate: 57.8%


In [8]:
# Focus on 4th down analysis (no 2-pt conversions found in data)
print("="*60)
print("4TH DOWN PASS PLAY ANALYSIS")
print("="*60)

fourth_down = supp[supp['down'] == 4].copy()

print(f"\nTotal 4th down pass plays: {len(fourth_down)}")
print(f"Completion rate: {fourth_down['is_complete'].mean()*100:.1f}%")

print("\n📊 PASS RESULTS:")
print(fourth_down['pass_result'].value_counts())

print("\n📊 COMPLETION RATE BY YARDS TO GO:")
ytg_stats = fourth_down.groupby('yards_to_go')['is_complete'].agg(['mean', 'count'])
ytg_stats = ytg_stats[ytg_stats['count'] >= 5].sort_index()
for ytg, row in ytg_stats.iterrows():
    print(f"  {ytg} yards: {row['mean']*100:.0f}% ({int(row['count'])} plays)")

4TH DOWN PASS PLAY ANALYSIS

Total 4th down pass plays: 491
Completion rate: 57.8%

📊 PASS RESULTS:
pass_result
C     284
I     187
IN     20
Name: count, dtype: int64

📊 COMPLETION RATE BY YARDS TO GO:
  1 yards: 71% (75 plays)
  2 yards: 61% (98 plays)
  3 yards: 64% (69 plays)
  4 yards: 61% (51 plays)
  5 yards: 60% (48 plays)
  6 yards: 52% (23 plays)
  7 yards: 72% (25 plays)
  8 yards: 33% (18 plays)
  9 yards: 50% (6 plays)
  10 yards: 47% (34 plays)
  12 yards: 40% (5 plays)
  17 yards: 0% (7 plays)


In [9]:
# =============================================================================
# 4TH DOWN: BEST ROUTES AND FORMATIONS
# =============================================================================
print("="*60)
print("4TH DOWN: BEST ROUTES (min 10 plays)")
print("="*60)

route_stats = fourth_down.groupby('route_of_targeted_receiver').agg({
    'is_complete': ['mean', 'count'],
    'yards_gained': 'mean'
}).round(2)
route_stats.columns = ['completion_rate', 'plays', 'avg_yards']
route_stats = route_stats[route_stats['plays'] >= 10].sort_values('completion_rate', ascending=False)
print(route_stats)

print("\n" + "="*60)
print("4TH DOWN: BEST FORMATIONS (min 10 plays)")
print("="*60)
form_stats = fourth_down.groupby('offense_formation').agg({
    'is_complete': ['mean', 'count'],
    'yards_gained': 'mean'
}).round(2)
form_stats.columns = ['completion_rate', 'plays', 'avg_yards']
form_stats = form_stats[form_stats['plays'] >= 10].sort_values('completion_rate', ascending=False)
print(form_stats)

print("\n" + "="*60)
print("4TH DOWN: BY COVERAGE TYPE")
print("="*60)
cov_stats = fourth_down.groupby('team_coverage_man_zone')['is_complete'].agg(['mean', 'count'])
print(cov_stats)

4TH DOWN: BEST ROUTES (min 10 plays)
                            completion_rate  plays  avg_yards
route_of_targeted_receiver                                   
ANGLE                                  0.85     13       7.85
FLAT                                   0.82     56       5.77
SLANT                                  0.77     43       8.47
CROSS                                  0.62     58       7.74
HITCH                                  0.61     72       6.12
OUT                                    0.59     83       5.55
IN                                     0.48     46       5.48
CORNER                                 0.41     27       8.59
POST                                   0.39     36       6.28
GO                                     0.30     53       7.04

4TH DOWN: BEST FORMATIONS (min 10 plays)
                   completion_rate  plays  avg_yards
offense_formation                                   
SINGLEBACK                    0.75     20       6.45
EMPTY             

In [10]:
# =============================================================================
# COACH RECOMMENDATIONS: 4TH DOWN
# =============================================================================
print("="*60)
print("📋 4TH DOWN COACH DECISION GUIDE")
print("="*60)

print("""
🏈 4TH DOWN PASS PLAYS (491 plays analyzed)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OVERALL: 57.8% completion rate

✅ HIGH-PERCENTAGE ROUTES (>75%):
   • ANGLE: 85% (13 plays) - Best option!
   • FLAT: 82% (56 plays) - Very reliable
   • SLANT: 77% (43 plays) - Good yards after catch

⚠️ MODERATE ROUTES (50-65%):
   • CROSS: 62% (58 plays)
   • HITCH: 61% (72 plays)
   • OUT: 59% (83 plays)

❌ AVOID THESE ROUTES (<50%):
   • IN: 48% (46 plays)
   • CORNER: 41% (27 plays)
   • POST: 39% (36 plays)
   • GO: 30% (53 plays) - Worst option!

BY YARDS TO GO:
   • 4th & Short (1-3): 65% completion - Use FLAT/SLANT
   • 4th & Medium (4-7): 59% completion - Use ANGLE/FLAT
   • 4th & Long (8+): 42% completion - Tough situation

FORMATION:
   • SINGLEBACK: 75% (best but limited sample)
   • EMPTY/SHOTGUN: ~57-58%

COVERAGE DOESN'T MATTER MUCH:
   • Man: 59.1%
   • Zone: 56.6%
""")

# Quick lookup table
print("\n📊 QUICK REFERENCE TABLE:")
print("-" * 50)
quick_ref = fourth_down.groupby(['route_of_targeted_receiver']).agg({
    'is_complete': 'mean',
    'yards_gained': 'mean',
    'game_id': 'count'
}).round(2)
quick_ref.columns = ['Comp%', 'Avg Yds', 'Plays']
quick_ref = quick_ref[quick_ref['Plays'] >= 10].sort_values('Comp%', ascending=False)
quick_ref['Comp%'] = (quick_ref['Comp%'] * 100).astype(int).astype(str) + '%'
print(quick_ref.to_string())

📋 4TH DOWN COACH DECISION GUIDE

🏈 4TH DOWN PASS PLAYS (491 plays analyzed)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OVERALL: 57.8% completion rate

✅ HIGH-PERCENTAGE ROUTES (>75%):
   • ANGLE: 85% (13 plays) - Best option!
   • FLAT: 82% (56 plays) - Very reliable
   • SLANT: 77% (43 plays) - Good yards after catch

⚠️ MODERATE ROUTES (50-65%):
   • CROSS: 62% (58 plays)
   • HITCH: 61% (72 plays)
   • OUT: 59% (83 plays)

❌ AVOID THESE ROUTES (<50%):
   • IN: 48% (46 plays)
   • CORNER: 41% (27 plays)
   • POST: 39% (36 plays)
   • GO: 30% (53 plays) - Worst option!

BY YARDS TO GO:
   • 4th & Short (1-3): 65% completion - Use FLAT/SLANT
   • 4th & Medium (4-7): 59% completion - Use ANGLE/FLAT
   • 4th & Long (8+): 42% completion - Tough situation

FORMATION:
   • SINGLEBACK: 75% (best but limited sample)
   • EMPTY/SHOTGUN: ~57-58%

COVERAGE DOESN'T MATTER MUCH:
   • Man: 59.1%
   • Zone: 56.6%


📊 QUICK REFERENCE TABLE:
--------------------------------------------------
       